In [ ]:
from datetime import datetime
import requests
from io import BytesIO
from PIL import Image

import pandas as pd
import numpy as np
from scipy.stats import percentileofscore

import nfl_data_py as nfl

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import plotly.colors as cl
from plotly.subplots import make_subplots

from resources.plotly_theme import nfl_template
from resources.heat_map import heat_map
from resources.get_nfl_data import get_pbp_data, get_team_info, get_matchups
from resources.team_stats import get_team_stats
from resources.player_stats import get_player_stats

from resources.game_review import production_by_qtr, production_by_down, receiver_sr_down_distance, rusher_sr_down_distance, epa_box_score

pio.templates['nfl_template'] = nfl_template

In [ ]:
''' Import Data '''

# Import
team_data = get_team_info()
pbp_data = get_pbp_data(years=[2025], include_postseason=False)

player_info = nfl.import_players()

## Data ##

# Offense
offense_stats = get_team_stats(pbp_data, unit='offense')

# Team Defense
defense_stats = get_team_stats(pbp_data, unit='defense')


print(player_info.head(2).to_string())

# Rushing

In [ ]:
# NOTE - No scrambles
run_data = pbp_data.loc[(pbp_data['rush'] == 1) & (pbp_data['qb_scramble'] == 0) &
                        (pbp_data['ydstogo'] > 0), :].copy()

fig = px.histogram(
    data_frame=run_data,
    x='ep'
)
fig.show()

In [ ]:
''' League Performance by Down & Distance '''

league_rush = run_data.groupby(['down', 'Distance']).aggregate(
    Plays=('rush', 'sum'),
    Attempts=('rush_attempt', 'sum'),
    Yards=('rushing_yards', 'sum'),
    Successes=('success', 'sum'),
    EPA=('epa', 'sum')
)
league_rush = league_rush.reindex(labels=['Short', 'Medium', 'Long'], level='Distance')

league_rush['Yds / Att'] = round(league_rush['Yards'] / league_rush['Attempts'], 2)
league_rush['Success Rate'] = round((league_rush['Successes'] / league_rush['Attempts']) * 100, 2)
league_rush['EPA / Play'] = round((league_rush['EPA'] / league_rush['Plays']), 2)

print(league_rush.to_string())

# EPA vs. yardstogo
fig = px.scatter(
    data_frame=run_data,
    x='ydstogo',
    y='epa',
    title='EPA vs. ydstogo',
    trendline='ols'
)
fig.show()

In [ ]:
''' League Performance by On Schedule or Not '''

on_schedule_rush = run_data.groupby(['On Schedule Play']).aggregate(
    Plays=('rush', 'sum'),
    Attempts=('rush_attempt', 'sum'),
    Yards=('rushing_yards', 'sum'),
    Successes=('success', 'sum'),
    EPA=('epa', 'sum')
)
on_schedule_rush['Yds / Att'] = round(on_schedule_rush['Yards'] / on_schedule_rush['Attempts'], 2)
on_schedule_rush['Success Rate'] = round((on_schedule_rush['Successes'] / on_schedule_rush['Attempts']) * 100, 2)
on_schedule_rush['EPA / Play'] = round((on_schedule_rush['EPA'] / on_schedule_rush['Plays']), 2)

print(on_schedule_rush.to_string())

In [ ]:
''' League Performance by Field Position '''

run_data['field pos'] = np.ceil(run_data['yardline_100'] / 10) * 10

field_pos_rushing = run_data.groupby(['field pos']).aggregate(
    Plays=('rush', 'sum'),
    Attempts=('rush_attempt', 'sum'),
    Yards=('rushing_yards', 'sum'),
    Successes=('success', 'sum'),
    EPA=('epa', 'sum')
)
field_pos_rushing['Yds / Att'] = round(field_pos_rushing['Yards'] / field_pos_rushing['Attempts'], 2)
field_pos_rushing['Success Rate'] = round((field_pos_rushing['Successes'] / field_pos_rushing['Attempts']) * 100, 2)
field_pos_rushing['EPA / Play'] = round((field_pos_rushing['EPA'] / field_pos_rushing['Plays']), 2)

print(field_pos_rushing.to_string())

# EPA vs. yardline_100
fig = px.scatter(
    data_frame=run_data,
    x='yardline_100',
    y='epa',
    title='EPA vs. yardline_100',
    trendline='ols'
)
fig.show()

In [ ]:
''' League Performance by Score Margin '''

run_data['score margin'] = run_data['posteam_score'] - run_data['defteam_score']

margin_rushing = run_data.groupby(['score margin']).aggregate(
    Plays=('rush', 'sum'),
    Attempts=('rush_attempt', 'sum'),
    Yards=('rushing_yards', 'sum'),
    Successes=('success', 'sum'),
    EPA=('epa', 'sum')
)
margin_rushing['Yds / Att'] = round(margin_rushing['Yards'] / margin_rushing['Attempts'], 2)
margin_rushing['Success Rate'] = round((margin_rushing['Successes'] / margin_rushing['Attempts']) * 100, 2)
margin_rushing['EPA / Play'] = round((margin_rushing['EPA'] / margin_rushing['Plays']), 2)

print(margin_rushing.head().to_string())

# EPA vs. margin
fig = px.scatter(
    data_frame=run_data,
    x='score margin',
    y='epa',
    title='EPA vs. score margin',
    trendline='ols'
)
fig.show()


In [ ]:
''' League Performance by EP '''

# EPA vs. EP
fig = px.scatter(
    data_frame=run_data,
    x='ep',
    y='epa',
    title='EPA vs. EP',
    trendline='ols'
)
fig.show()

# Success vs. EP
fig = px.scatter(
    data_frame=run_data,
    x='ep',
    y='success',
    title='Success vs. EP',
    trendline='ols'
)
fig.show()

In [ ]:
single_rush_corr = run_data[['ep', 'ydstogo', 'yardline_100', 'score margin', 'On Schedule Play', 'yards_gained', 'epa', 'success']].corr()
fig = px.imshow(
    img=single_rush_corr,
    x=single_rush_corr.columns,
    y=single_rush_corr.index,
    title='Individual rushes',
    text_auto='.1%'
)
fig.show()


# run_data['ydstogo perc'] = run_data['ydstogo'].rank(method='max', ascending=False, pct=True)
# run_data['yardline_100 perc'] = run_data['yardline_100'].rank(method='max', ascending=False, pct=True)
# run_data['score margin perc'] = run_data['score margin'].rank(method='max', ascending=True, pct=True)
# run_data['ep perc'] = run_data['ep'].rank(method='max', ascending=True, pct=True)
# run_data['situation score'] = run_data[['ep perc', 'ydstogo perc']].mean(axis=1)
# run_data['situation score'] = (run_data['ep perc'] * 0.25) + (run_data['ydstogo perc'] * 0.75)
# run_data['situation score'] = (run_data['ydstogo perc'] * 0.75) + (run_data['yardline_100 perc'] * 0.25)
# run_data['situation score'] = (run_data['On Schedule Play'] * 0.25) + (run_data['ydstogo perc'] * 0.75)
run_data['situation score'] = (run_data['ydstogo perc'] * 0.75) + (run_data['yardline_100 perc'] * 0.125) + (run_data['score margin perc'] * .125)
# run_data['situation score'] = (run_data['On Schedule Play'] * 0.75) + (run_data['yardline_100 perc'] * 0.125) + (run_data['score margin perc'] * .125)

single_rush_corr = run_data[['situation score', 'yards_gained', 'epa', 'success']].corr()
fig = px.imshow(
    img=single_rush_corr,
    x=single_rush_corr.columns,
    y=single_rush_corr.index,
    title='Individual rushes',
    text_auto='.1%'
)
fig.show()

In [ ]:
''' By Rusher '''

by_rusher = run_data.groupby(['posteam', 'rusher']).aggregate(
    player_id=('rusher_player_id', 'first'),
    Plays=('rush', 'sum'),
    Attempts=('rush_attempt', 'sum'),
    Yards=('rushing_yards', 'sum'),
    TDs=('touchdown', 'sum'),
    Fumbles=('fumble_lost', 'sum'),
    FirstDowns=('first_down', 'sum'),
    Successes=('success', 'sum'),
    EPA=('epa', 'sum'),
    AvgYdsToGo=('ydstogo', 'mean'),
    MedianEP=('ep', 'median'),
    AvgEP=('ep', 'mean'),
    OnScheduleAtts=('On Schedule Play', lambda x: x[run_data['rush_attempt'] == 1].sum()),
    LongDownAtts=('rush', lambda x: x[(run_data['down'] != 1) & (run_data['Distance'] == 'Long')].sum()),
).sort_values(by=['Attempts'], ascending=False)

by_rusher['Yds / Att'] = round(by_rusher['Yards'] / by_rusher['Attempts'], 2)
by_rusher['Success Rate'] = round((by_rusher['Successes'] / by_rusher['Attempts']) * 100, 2)
by_rusher['EPA / Play'] = round((by_rusher['EPA'] / by_rusher['Plays']), 2)
by_rusher['1D Rate'] = round((by_rusher['FirstDowns'] / by_rusher['Attempts']) * 100, 2)
by_rusher['TD Rate'] = round((by_rusher['TDs'] / by_rusher['Attempts']) * 100, 2)
by_rusher['On Schedule %'] = round((by_rusher['OnScheduleAtts'] / by_rusher['Attempts']) * 100, 2)
by_rusher['Long Down %'] = round((by_rusher['LongDownAtts'] / by_rusher['Attempts']) * 100, 2)

by_rusher = by_rusher.loc[by_rusher['Attempts'] >= 40, :]

by_rusher['EP Perc'] = by_rusher['MedianEP'].rank(method='max', pct=True)
by_rusher['On Schedule Perc'] = by_rusher['On Schedule %'].rank(method='max', pct=True)
# by_rusher['Situation Score'] = by_rusher[['EP Perc', 'On Schedule Perc']].mean(axis=1)
by_rusher['Situation Score'] = (by_rusher['EP Perc'] * 0.75) + (by_rusher['On Schedule Perc'] * 0.25)

print(by_rusher.head().to_string())

In [ ]:
''' Correlations '''

print(f'Performance over time')
player_corr = by_rusher[['On Schedule %', 'AvgYdsToGo', 'MedianEP', 'AvgEP', 'Situation Score', 'Yds / Att', 'Success Rate', 'EPA / Play']].corr()
fig = px.imshow(
    img=player_corr,
    x=player_corr.columns,
    y=player_corr.index,
    title='Performance over time',
    text_auto='.1%'
)
fig.show()
# print(player_corr.to_string())



In [ ]:


# Rusher Success Rate vs. Median EP
fig = px.scatter(
    data_frame=by_rusher,
    x='MedianEP',
    y='Success Rate',
    title='Rusher Success Rate vs. Median EP',
    trendline='ols'
)
fig.show()

# Rusher EPA / Play vs. Avg Yds To Go
fig = px.scatter(
    data_frame=by_rusher,
    x='AvgYdsToGo',
    y='EPA / Play',
    title='Rusher EPA / Play vs. Avg Yds To Go',
    trendline='ols'
)
fig.show()

# Passing

In [ ]:
# NOTE - No scrambles
pass_data = pbp_data.loc[(pbp_data['pass'] == 1) & (pbp_data['qb_scramble'] == 0) &
                        (pbp_data['ydstogo'] > 0), :].copy()

fig = px.histogram(
    data_frame=pass_data,
    x='ep'
)
fig.show()

In [ ]:
''' League Performance by Down & Distance '''

down_dist_passing = pass_data.groupby(['down', 'Distance']).aggregate(
    Plays=('pass', 'sum'),
    Attempts=('pass_attempt', 'sum'),
    Yards=('passing_yards', 'sum'),
    Successes=('success', 'sum'),
    EPA=('epa', 'sum')
)
down_dist_passing = down_dist_passing.reindex(labels=['Short', 'Medium', 'Long'], level='Distance')

down_dist_passing['Yds / Att'] = round(down_dist_passing['Yards'] / down_dist_passing['Attempts'], 2)
down_dist_passing['Success Rate'] = round((down_dist_passing['Successes'] / down_dist_passing['Attempts']) * 100, 2)
down_dist_passing['EPA / Play'] = round((down_dist_passing['EPA'] / down_dist_passing['Plays']), 2)

print(down_dist_passing.to_string())

# EPA vs. yardstogo
fig = px.scatter(
    data_frame=pass_data,
    x='ydstogo',
    y='epa',
    title='EPA vs. ydstogo',
    trendline='ols'
)
fig.show()

In [ ]:
''' League Performance by On Schedule or Not '''

on_schedule_pass = pass_data.groupby(['On Schedule Play']).aggregate(
    Plays=('pass', 'sum'),
    Attempts=('pass_attempt', 'sum'),
    Yards=('passing_yards', 'sum'),
    Successes=('success', 'sum'),
    EPA=('epa', 'sum')
)
on_schedule_pass['Yds / Att'] = round(on_schedule_pass['Yards'] / on_schedule_pass['Attempts'], 2)
on_schedule_pass['Success Rate'] = round((on_schedule_pass['Successes'] / on_schedule_pass['Attempts']) * 100, 2)
on_schedule_pass['EPA / Play'] = round((on_schedule_pass['EPA'] / on_schedule_pass['Plays']), 2)

print(on_schedule_pass.to_string())

In [ ]:
''' League Performance by Field Position '''

pass_data['field pos'] = np.ceil(pass_data['yardline_100'] / 10) * 10

field_pos_passing = pass_data.groupby(['field pos']).aggregate(
    Plays=('pass', 'sum'),
    Attempts=('pass_attempt', 'sum'),
    Yards=('passing_yards', 'sum'),
    Successes=('success', 'sum'),
    EPA=('epa', 'sum')
)
field_pos_passing['Yds / Att'] = round(field_pos_passing['Yards'] / field_pos_passing['Attempts'], 2)
field_pos_passing['Success Rate'] = round((field_pos_passing['Successes'] / field_pos_passing['Attempts']) * 100, 2)
field_pos_passing['EPA / Play'] = round((field_pos_passing['EPA'] / field_pos_passing['Plays']), 2)

print(field_pos_passing.to_string())

# EPA vs. yardline_100
fig = px.scatter(
    data_frame=pass_data,
    x='yardline_100',
    y='epa',
    title='EPA vs. yardline_100',
    trendline='ols'
)
fig.show()

In [ ]:
''' League Performance by Score Margin '''

pass_data['score margin'] = pass_data['posteam_score'] - pass_data['defteam_score']

margin_passing = pass_data.groupby(['score margin']).aggregate(
    Plays=('pass', 'sum'),
    Attempts=('pass_attempt', 'sum'),
    Yards=('passing_yards', 'sum'),
    Successes=('success', 'sum'),
    EPA=('epa', 'sum')
)
margin_passing['Yds / Att'] = round(margin_passing['Yards'] / margin_passing['Attempts'], 2)
margin_passing['Success Rate'] = round((margin_passing['Successes'] / margin_passing['Attempts']) * 100, 2)
margin_passing['EPA / Play'] = round((margin_passing['EPA'] / margin_passing['Plays']), 2)

print(margin_passing.head().to_string())

# EPA vs. margin
fig = px.scatter(
    data_frame=pass_data,
    x='score margin',
    y='epa',
    title='EPA vs. score margin',
    trendline='ols'
)
fig.show()


In [ ]:
''' League Performance by EP '''

# EPA vs. EP
fig = px.scatter(
    data_frame=pass_data,
    x='ep',
    y='epa',
    title='EPA vs. EP',
    trendline='ols'
)
fig.show()

# Success vs. EP
fig = px.scatter(
    data_frame=pass_data,
    x='ep',
    y='success',
    title='Success vs. EP',
    trendline='ols'
)
fig.show()

In [ ]:
single_pass_corr = pass_data[['ep', 'ydstogo', 'yardline_100', 'score margin', 'On Schedule Play', 'yards_gained', 'epa', 'success']].corr()
fig = px.imshow(
    img=single_pass_corr,
    x=single_pass_corr.columns,
    y=single_pass_corr.index,
    title='Individual Passes',
    text_auto='.1%'
)
fig.show()


pass_data['ydstogo perc'] = pass_data['ydstogo'].rank(method='max', ascending=False, pct=True)
pass_data['yardline_100 perc'] = pass_data['yardline_100'].rank(method='max', ascending=False, pct=True)
pass_data['score margin perc'] = pass_data['score margin'].rank(method='max', ascending=True, pct=True)
pass_data['ep perc'] = pass_data['ep'].rank(method='max', ascending=True, pct=True)
# pass_data['situation score'] = pass_data[['ep perc', 'ydstogo perc']].mean(axis=1)
# pass_data['situation score'] = (pass_data['ep perc'] * 0.25) + (pass_data['ydstogo perc'] * 0.75)
# pass_data['situation score'] = (pass_data['ydstogo perc'] * 0.75) + (pass_data['yardline_100 perc'] * 0.25)
# pass_data['situation score'] = (pass_data['On Schedule Play'] * 0.25) + (pass_data['ydstogo perc'] * 0.75)
# pass_data['situation score'] = (pass_data['ydstogo perc'] * 0.75) + (pass_data['yardline_100 perc'] * 0.125) + (pass_data['score margin perc'] * .125)
# pass_data['situation score'] = (pass_data['On Schedule Play'] * 0.75) + (pass_data['yardline_100 perc'] * 0.125) + (pass_data['score margin perc'] * .125)

single_pass_corr = pass_data[['situation score', 'yards_gained', 'epa', 'success']].corr()
fig = px.imshow(
    img=single_pass_corr,
    x=single_pass_corr.columns,
    y=single_pass_corr.index,
    title='Individual passes',
    text_auto='.1%'
)
fig.show()